# Train an image CLASSIFIER on Colab (free GPU): beard / no_beard

For datasets in **folder layout**: `train/<class>/*.jpg`, `val/<class>/*.jpg`, `test/<class>/*.jpg`.
It learns to say "beard" or "no_beard" from a cropped FACE picture (the pipeline finds the face first).

**Before you start**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Upload `beard_cls_dataset.zip` to Google Drive (My Drive).
3. Run the cells top to bottom.

**When it finishes** the notebook downloads one `.pt` file. Copy it to `D:\priya\Live CCTV analytics\models\`.

In [ ]:
# 1) GPU check
import shutil, subprocess
if shutil.which("nvidia-smi") is None:
    raise SystemExit("NO GPU FOUND -> Runtime > Change runtime type > T4 GPU > Save, then Runtime > Restart session and run again")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

In [ ]:
# 2) Install the same Ultralytics version used on your PC (so the model loads there)
!pip -q install "ultralytics==8.4.176"
import ultralytics; print("ultralytics", ultralytics.__version__)

In [ ]:
# 3) SETTINGS  <-- edit this cell
DATASET_ZIP = "/content/drive/MyDrive/beard_cls_dataset.zip"   # path of the zip on Google Drive
USE_DRIVE   = True       # True: read the zip from Google Drive.  False: you will be asked to upload it
OUTPUT_NAME = "beard_cls.pt"   # file name you will download

EPOCHS = 40              # training stops early if it stops improving (patience)
IMGSZ  = 224             # classifier picture size
BATCH  = 32
BASE   = "yolo11n-cls.pt"  # small pretrained classifier: fast on your CPU PC

In [ ]:
# 4) Get the zip
import os, shutil
ZIP_LOCAL = "/content/dataset.zip"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.exists(DATASET_ZIP), f"Zip not found: {DATASET_ZIP}  (check the name/path in cell 3)"
    shutil.copy(DATASET_ZIP, ZIP_LOCAL)
else:
    from google.colab import files
    up = files.upload()
    shutil.move(next(iter(up)), ZIP_LOCAL)
print("zip size (MB):", round(os.path.getsize(ZIP_LOCAL) / 1e6, 1))

In [ ]:
# 5) Unzip and check the folders
import zipfile, glob, collections
from pathlib import Path
ROOT = Path("/content/dataset")
if ROOT.exists():
    shutil.rmtree(ROOT)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    bad = [n for n in z.namelist() if n.startswith("/") or ".." in Path(n).parts]
    assert not bad, f"unsafe paths in zip: {bad[:3]}"
    z.extractall(ROOT)
for split in ("train", "val", "test"):
    counts = {p.name: len(list(p.glob("*"))) for p in sorted((ROOT / split).iterdir()) if p.is_dir()}
    print(split, counts)
assert (ROOT / "train").exists() and (ROOT / "val").exists(), "need train/ and val/ folders"
CLASSES = sorted(p.name for p in (ROOT / "train").iterdir() if p.is_dir())
print("classes:", CLASSES)

In [ ]:
# 6) Train (a few minutes on a T4)
from ultralytics import YOLO
model = YOLO(BASE)
results = model.train(
    data=str(ROOT), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, workers=2, patience=15,
    project="/content/runs", name="beard_cls", exist_ok=True,
    # make it cope with the small, soft, off-angle faces a CCTV gives
    hsv_h=0.02, hsv_s=0.5, hsv_v=0.5, degrees=15, translate=0.1, scale=0.4, fliplr=0.5, erasing=0.2,
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
print("best weights:", BEST)

In [ ]:
# 7) Score on the TEST pictures the model has never seen (look at every class)
m = YOLO(str(BEST))
tot = ok = 0
print(f"{'class':12s} {'pictures':>8s} {'correct':>8s} {'accuracy':>9s}")
wrong = collections.Counter()
for cls in CLASSES:
    files_ = sorted(glob.glob(str(ROOT / "test" / cls / "*")))
    good = 0
    for r in m.predict(files_, imgsz=IMGSZ, device=0, verbose=False, stream=True):
        pred = r.names[r.probs.top1]
        if pred == cls: good += 1
        else: wrong[(cls, pred)] += 1
    print(f"{cls:12s} {len(files_):8d} {good:8d} {good / max(1, len(files_)):9.3f}")
    tot += len(files_); ok += good
print(f"{'ALL':12s} {tot:8d} {ok:8d} {ok / max(1, tot):9.3f}")
print("mistakes (true class, predicted):", dict(wrong))

In [ ]:
# 8) Save + download
OUT = Path("/content") / OUTPUT_NAME
shutil.copy(BEST, OUT)
if USE_DRIVE:
    shutil.copy(OUT, Path("/content/drive/MyDrive") / OUTPUT_NAME)   # backup copy in your Drive
    print("backup saved in Google Drive:", OUTPUT_NAME)
from google.colab import files
files.download(str(OUT))

## After the download (on your PC)

1. Copy the `.pt` file into `D:\priya\Live CCTV analytics\models\`.
2. Check it: `python tools\inspect_model.py models\beard_cls.pt` (a classifier lists its two classes).
3. Tell Claude the cell 7 table. The pipeline still needs a small code change to use a classifier on faces found by YuNet.